# One-time bridge: Google Drive <-> Kaggle Dataset

Your training progress lives on **Drive** (Colab), but the Kaggle twin of the production
notebook (`colab_production_kaggle.ipynb`) persists everything in a **private Kaggle dataset**
`<USER>/silverwing-state`. This notebook moves state between the two platforms.

**Prerequisites:**
1. A Kaggle account with phone verification (needed for API uploads).
2. API token: kaggle.com > avatar > Settings > API > **Create New Token** -> downloads `kaggle.json`.

**When to run what (on Colab, CPU runtime is enough):**
- Cell 3 (**push**) once before your first Kaggle session - seeds your current Drive progress.
- Cell 4 (**pull**) after Kaggle sessions advanced beyond Colab - brings newest checkpoints back to Drive so a Colab session can resume them.

In [ ]:
# Cell 1: Mount Google Drive + Kaggle credentials
import os

import shutil

from google.colab import drive, files

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print('Upload kaggle.json - kaggle.com > Settings > API > Create New Token')
    up = files.upload()
    name = next(iter(up))
    shutil.move(name, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
!pip install -q kaggle
print('Kaggle credentials ready')

In [ ]:
# Cell 2: State-dataset helpers (one private dataset holds everything)
import json
import re
import shutil
import subprocess
from pathlib import Path

USER = 'videlisndichi'  # <-- EDIT ME
STATE = f'{USER}/silverwing-state'
STAGE = Path('/content/.stage')
SUBDIRS = ['checkpoints/pretrain', 'checkpoints/sft',
           'corpus/corpus-external', 'tokenizer/tokenizer-v2']


def _kaggle(*args):
    return subprocess.run(['kaggle', *args], capture_output=True, text=True)


def _stage_from(src_root, keep_last=4):
    """Mirror the Drive layout into STAGE, pruning ancient step checkpoints."""
    if STAGE.exists():
        shutil.rmtree(STAGE)
    copied = []
    for sub in SUBDIRS:
        src = Path(src_root) / sub
        if not src.exists():
            continue
        dst = STAGE / sub
        dst.mkdir(parents=True)
        files = sorted(f for f in src.iterdir() if f.is_file())
        steps = [f for f in files if re.match(r'step-\d+\.pt$', f.name)]
        others = [f for f in files if f not in steps]
        steps = sorted(steps, key=lambda p: int(p.stem.split('-')[1]))[-keep_last:]
        for f in others + steps:
            shutil.copy2(f, dst / f.name)
        copied.append(sub)
    meta = {'title': 'Silverwing State', 'id': STATE,
            'licenses': [{'name': 'CC0-1.0'}]}
    (STAGE / 'dataset-metadata.json').write_text(json.dumps(meta))
    return copied

In [ ]:
# Cell 3: PUSH Drive -> Kaggle (seed before first Kaggle session;
# safe to re-run whenever you want to overwrite dataset state from Drive)
copied = _stage_from(DRIVE)
assert copied, f'nothing found under {DRIVE} - check DRIVE path'
if _kaggle('datasets', 'status', STATE).returncode == 0:
    r = _kaggle('datasets', 'version', '-p', str(STAGE), '-m',
                'seed/update from Drive', '--dir-mode', 'zip')
else:
    r = _kaggle('datasets', 'create', '-p', str(STAGE), '--dir-mode', 'zip')
print(r.stdout[-800:], r.stderr[-800:])
assert r.returncode == 0, 'push failed - see output above'
size = sum(f.stat().st_size for f in STAGE.rglob('*') if f.is_file())
print(f'Pushed {len(copied)} dirs ({size / 1e9:.2f} GB staged): {copied}')
print('Dataset:', f'https://www.kaggle.com/datasets/{STATE}')

In [ ]:
# Cell 4: PULL Kaggle -> Drive (after Kaggle advanced beyond Colab)
dl = Path('/content/.dl')
if dl.exists():
    shutil.rmtree(dl)
dl.mkdir(parents=True)
r = _kaggle('datasets', 'download', STATE, '-p', str(dl), '--unzip')
print(r.stdout[-500:], r.stderr[-500:])
assert r.returncode == 0, 'download failed'
for sub in SUBDIRS:
    src = dl / sub
    if not src.exists():
        continue
    dst = Path(DRIVE) / sub
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file():
            shutil.copy2(f, dst / f.name)
    print('restored to Drive:', sub)
shutil.rmtree(dl)